# Baseline v2
## Upgrades
- incorporate the custom metric: Cumulated Profit and Loss (PnL)
- use 4 CV folds instead of 3
- no hyperparameter tuning

### Get the data

In [1]:
# # for plotly plots inside nb
# %pip install "nbformat>=4.2.0"

In [2]:
import pandas as pd

# Load the dictionary saved in the pickle file.
setup = pd.read_pickle("../data/cv_setup.pkl")

# Extract the two pandas DataFrames and the list of CV splits.
train = setup["train_data"]
validation = setup["validation_data"]
cv_splits = setup["cv_splits"]


### Cumulative PnL

In [3]:
import numpy as np
import plotly.graph_objects as go

# Index imbalances by date so each validation row gets the correct value.
imbalances = pd.read_csv(
    "../data/raw/imbalances.csv",
    parse_dates=["date"],
).set_index("date")["imbalances"]


def calc_pnl_per_period(estimator, X_valid, y_valid):
    """Return one PnL value per validation timestamp."""
    dates = pd.to_datetime(X_valid["date"])
    forecasts = estimator.predict(X_valid)

    # Match each forecast to the imbalance for the same 15-minute period.
    matched_imbalances = imbalances.reindex(dates).to_numpy()
    if not np.isfinite(matched_imbalances).all():
        raise ValueError("Missing or invalid imbalances for validation dates.")

    # Buy 50 MW for a nonnegative forecast, otherwise sell 50 MW.
    positions = np.where(forecasts >= 0, 50, -50)

    # Compute each observation's profit or loss.
    pnl = positions * np.asarray(y_valid)

    # Skip periods with excessive imbalance.
    pnl = np.where(np.abs(matched_imbalances) > 1000, 0, pnl)

    # Return PnL per every observation
    return pd.Series(pnl, index=dates, name="pnl")

def pnl_scorer(estimator, X_valid, y_valid):
    # cross_validate expects one score per fold, so sum its 15-minute PnLs.
    return calc_pnl_per_period(estimator, X_valid, y_valid).sum()


def plot_cumulative_pnl(cv_results, cv_splits, X, y, renderer=None):
    """Plot cumulative validation PnL in euros over time.

    Each point shows the running total from the model's forecasts and the
    observed spreads, using the competition's trading rule. Returns a table
    with each 15-minute period's PnL and its cumulative PnL.
    """
    # Collect one timestamp-indexed PnL Series per validation fold.
    pnl_by_fold = []

    # Pair each fitted model with its fold. Each split contains (train_idx, valid_idx).
    # The model is already trained, so we ignore train_idx (the _) and use valid_idx.
    for model, (_, valid_idx) in zip(cv_results["estimator"], cv_splits):
        # Gives the features for this fold's validation rows.
        valid_features = X.iloc[valid_idx]

        # Gives the spreads for the same fold's validation rows.
        valid_spreads = y.iloc[valid_idx]

        # The function predicts with the model and calculates PnL for each 15-minute period.
        period_pnl = calc_pnl_per_period(model, valid_features, valid_spreads)

        # Save this fold's PnL values so we can later combine all folds into one curve.
        pnl_by_fold.append(period_pnl)

    # Stack the PnL Series from all folds into one long Series (pd.concat).
    # Sort by timestamp, oldest first (sort_index).
    # Turn the Series into a DataFrame with one column named "pnl" (to_frame).
    results = pd.concat(pnl_by_fold).sort_index().to_frame(name="pnl")

    # Add a column showing total PnL up to and including each timestamp.
    # For example, period PnLs [100, -30, 50] give cumulative PnLs [100, 70, 120].
    results["pnl_cum"] = results["pnl"].cumsum()

    # Create an empty Plotly chart.
    fig = go.Figure()

    # Put timestamps on the x-axis and cumulative PnL on the y-axis.
    # mode="lines" connects consecutive points so you can follow the total over time.
    fig.add_scatter(x=results.index, y=results["pnl_cum"], mode="lines")

    # Set the text displayed on the chart.
    fig.update_layout(
        title="Cumulative validation PnL", # Title above the chart.
        xaxis_title="Date",                # Label below the horizontal axis.
        yaxis_title="Cumulative PnL (€)",  # Label beside the vertical axis.
    )

    # Display the chart. Passing renderer="browser" opens it in your browser.
    fig.show(renderer=renderer)

    print(f"Total validation PnL: €{results['pnl_cum'].iloc[-1]:,.2f}")
    return results


### Performance evaluation

In [4]:
from IPython.display import display

def show_cv_scores(cv_results, *, show_plot=False, cv_splits=None,
                   X=None, y=None, renderer=None):
    """Show scores for each validation fold and optionally plot the 15-minute PnL curve."""
    if show_plot:
        if cv_splits is None or X is None or y is None:
            raise ValueError("For the plot, pass cv_splits, X and y from the same CV run.")
        if "estimator" not in cv_results:
            raise ValueError("For the plot, rerun cross_validate with return_estimator=True.")

    # Each row contains the scores for one fold.
    scores = pd.DataFrame({
        # scikit-learn returns negative MAE, the minus sign converts it back to positive MAE, where lower values indicate better predictions.
        "MAE": -cv_results["test_mae"],
        "RMSE": -cv_results["test_rmse"],   # same here: scikit-learn returns negative RMSE
        "Fold PnL (€)": cv_results["test_pnl"],
    })
    scores.index = pd.RangeIndex(1, len(scores) + 1, name="Fold")

    # Running total of PnL through the end of each fold.
    scores["Cumulative PnL (€)"] = scores["Fold PnL (€)"].cumsum()

    # Display rounded values without requiring jinja2.
    display(scores.round(2))

    if show_plot:
        plot_cumulative_pnl(cv_results, cv_splits, X, y, renderer=renderer)

    return scores


### Fit the baseline CatBoostRegressor

In [5]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor

def add_time_features(df):
    df = df.copy()  # creates an independent DataFrame so the function doesn't modify the original data
    df['hour'] = df['date'].dt.hour
    df['weekday'] = df['date'].dt.day_of_week
    df['month'] = df['date'].dt.month
    return df

preprocessor = Pipeline([
    ('time_features', FunctionTransformer(add_time_features)),
    ('select_features', ColumnTransformer([
        ('features', 'passthrough',
         ['wind', 'solar', 'load', 'hour', 'weekday', 'month'])
    ]))
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', CatBoostRegressor(verbose=False)) # disables training progress messages from CatBoost
])

# Split features and target
X = train.drop(columns=['spread'])
y = train['spread']

cv_results = cross_validate( # cv_reults is a dictionary containing the cross-validation results for each of your 3 time-series splits.
    pipeline, # the data preprocessing and the CatBoostRegressor model
    X,
    y, # the target variable
    cv=cv_splits,
    scoring={
        'mae': 'neg_mean_absolute_error', # is the negative Mean Absolute Error (MAE), measuring the average absolute difference between predictions and actual values
        'rmse': 'neg_root_mean_squared_error', # is the negative RMSE, measuring prediction error while penalizing large errors more heavily than MAE.
        "pnl": pnl_scorer,
    },
    return_estimator=True, # return models for each fold
)


### Baseline model performance

In [6]:
# Table and plot — use the same splits you passed to cross_validate.
fold_scores = show_cv_scores(cv_results, show_plot=True, cv_splits=cv_splits, X=X, y=y)


,MAE,RMSE,Fold PnL (€),Cumulative PnL (€)
Fold,,,,
1,57.13,109.32,4426454.0,4426454.0
2,96.27,177.05,5870439.0,10296893.0
3,164.71,241.23,16343650.5,26640543.5
4,237.87,353.20,15914358.5,42554902.0


Total validation PnL: €42,554,902.00
